# Security telemetry — Issue A (inference abuse / probing)

This notebook **queries a live Prometheus** instance (course VM or local port-forward) to assess whether **HTTP-level abuse** of `/recommend/<userid>` is *observable* via existing metrics.

**Prerequisites:** Prometheus scraping the recommendation service (`127.0.0.1:8082/metrics`) per [`monitoring/prometheus.yml`](https://github.com/cmu-seai/group-project-s26-model-impossible-1/blob/main/monitoring/prometheus.yml).

**Configure:** set `PROMETHEUS_URL` (default `http://127.0.0.1:9090`).

In [ ]:
import os
import json
from urllib.request import urlopen, Request
from urllib.parse import urlencode

PROMETHEUS_URL = os.environ.get("PROMETHEUS_URL", "http://127.0.0.1:9090").rstrip("/")


def promql_instant(query: str) -> dict:
    """Run instant PromQL; returns parsed JSON or raises."""
    q = urlencode({"query": query})
    url = f"{PROMETHEUS_URL}/api/v1/query?{q}"
    req = Request(url, headers={"Accept": "application/json"})
    with urlopen(req, timeout=30) as resp:
        return json.loads(resp.read().decode())


def print_vector_result(title: str, data: dict) -> None:
    print(f"\n=== {title} ===")
    if data.get("status") != "success":
        print("ERROR:", data)
        return
    rtype = data["data"].get("resultType")
    rows = data["data"].get("result") or []
    if not rows:
        print("(no series — metric absent or no samples in lookback)")
        return
    for row in rows[:50]:
        m = row.get("metric", {})
        v = row.get("value", [None, "?"])[1]
        print(f"  {m}  =>  {v}")
    if len(rows) > 50:
        print(f"  ... ({len(rows) - 50} more)")

## Queries tied to Issue A

- **Request rate** — spikes may indicate scanning or DoS-style load on the model path.
- **Errors / latency** — secondary signals (also see [`monitoring/alerts.yml`](../monitoring/alerts.yml)).
- **Model mix** — `sum by (model_type)` cannot prove an attack but can show path skew vs baseline.

**Negative result:** Prometheus does not label **client IP**; a burst cannot be attributed to “attacker” vs grader without access logs.

In [ ]:
QUERIES = [
    (
        "Total HTTP rec request rate (1m)",
        "sum(rate(recommendation_requests_total[1m]))",
    ),
    (
        "Request rate by model_type (1m)",
        "sum by (model_type) (rate(recommendation_requests_total[1m]))",
    ),
    (
        "Error rate (1m)",
        "sum(rate(recommendation_errors_total[1m]))",
    ),
    (
        "p95 latency seconds (5m window)",
        "histogram_quantile(0.95, sum(rate(recommendation_latency_seconds_bucket[5m])) by (le))",
    ),
    (
        "Kafka online rec events / min (exporter)",
        "online_rec_rate_per_minute",
    ),
]

for title, q in QUERIES:
    try:
        print_vector_result(title, promql_instant(q))
    except Exception as e:
        print(f"\n=== {title} ===")
        print(f"FAILED ({e}). Is Prometheus reachable at {PROMETHEUS_URL}?")

## Interpretation (paste into report)

1. If **request rate** is steady and **p95 latency** / **errors** are low, that is a **negative finding** for *observable* acute abuse in the sampled instant (not proof of absence).
2. If **Kafka `online_rec_rate_per_minute`** is near zero but HTTP rates are high, HTTP traffic may be **synthetic** (graders, scripts) rather than organic Kafka stream — still not proof of malice.
3. Record the **Prometheus “Evaluation time”** from the UI or add `time=` parameter to `/api/v1/query` for a historical snapshot when investigating an incident.